In [ ]:

#   FULL METAGENOMICS ANALYSIS PIPELINE 

# INSTALL 
# %pip install scikit-learn xgboost shap seaborn matplotlib joblib pandas numpy scipy scikit-bio statsmodels

# IMPORTS

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import distance
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import shap
import joblib


# USER CONFIG

CSV_PATH = ("C:\\Users\\Suprise Baloyi\\Downloads\\synthetic_metagenome.csv")  
TARGET_COL = "Delivery_Mode"           
ID_COL = "Infant_Number"

META_COLS = ["Delivery_Mode", "Feeding_Mode", "Age_days"]
SPECIES_PREFIX = "Species_"
OUTDIR = "analysis_outputs"

os.makedirs(OUTDIR, exist_ok=True)


# LOAD DATA

df = pd.read_csv(CSV_PATH)
species_cols = [c for c in df.columns if c.startswith(SPECIES_PREFIX)]
meta = df[[ID_COL] + META_COLS].copy()


# Convert Delivery Mode to Binary (0 = Vaginal, 1 = Caesarean)

meta[TARGET_COL] = meta[TARGET_COL].str.lower().map({
    "vaginal": 0,
    "normal": 0,
    "vaginal birth": 0,
    "cesarean": 1,
    "caesarean": 1,
    "c-section": 1
})

if meta[TARGET_COL].isna().any():
    raise ValueError("ERROR: Delivery_Mode contains unknown labels. Please check your CSV.")

y = meta[TARGET_COL]
X_raw = df[species_cols]


# FILTER LOW ABUNDANCE TAXA

min_prev = 0.01
min_mean = 1e-5

prev = (X_raw > 0).sum(axis=0) / X_raw.shape[0]
mean_ab = X_raw.mean(axis=0)

keep = (prev >= min_prev) & (mean_ab >= min_mean)
X_filtered = X_raw.loc[:, keep]

print(f"Filtered from {len(X_raw.columns)} → {len(X_filtered.columns)} species")


# TRANSFORMATIONS

# TSS normalization
X_tss = X_filtered.div(X_filtered.sum(axis=1), axis=0).fillna(0)
X_tss.to_csv(f"{OUTDIR}/X_TSS.csv")

# CLR transform
pseudocount = 1e-6
clr = np.log(X_tss + pseudocount)
X_clr = clr.sub(clr.mean(axis=1), axis=0)
X_clr.to_csv(f"{OUTDIR}/X_CLR.csv")


# ALPHA DIVERSITY

def shannon(x):
    p = x[x > 0]
    p = p / p.sum()
    return -(p * np.log(p)).sum()

def simpson(x):
    p = x[x > 0]
    p = p / p.sum()
    return 1 - (p**2).sum()

meta["Shannon"] = X_tss.apply(shannon, axis=1)
meta["Simpson"] = X_tss.apply(simpson, axis=1)

meta.to_csv(f"{OUTDIR}/metadata_with_alpha.csv", index=False)

plt.figure(figsize=(6,4))
sns.boxplot(x=y, y=meta["Shannon"])
plt.title("Shannon Diversity by Delivery Mode")
plt.xlabel("Delivery Mode (0 = Vaginal, 1 = Caesarean)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/alpha_diversity_shannon.png")
plt.close()


# BETA DIVERSITY (Bray–Curtis PCoA)

D = distance.squareform(distance.pdist(X_tss, metric="braycurtis"))

# Classical PCoA
n = D.shape[0]
H = np.eye(n) - np.ones((n,n))/n
B = -0.5 * H @ (D**2) @ H
eigvals, eigvecs = np.linalg.eigh(B)

idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

PCo1 = eigvecs[:, 0] * np.sqrt(eigvals[0])
PCo2 = eigvecs[:, 1] * np.sqrt(eigvals[1])

pcoa_df = pd.DataFrame({"PCo1": PCo1, "PCo2": PCo2, TARGET_COL: y})
pcoa_df.to_csv(f"{OUTDIR}/pcoa_coordinates.csv", index=False)

plt.figure(figsize=(6,5))
sns.scatterplot(data=pcoa_df, x="PCo1", y="PCo2", hue=TARGET_COL)
plt.title("PCoA (Bray–Curtis)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/pcoa_plot.png")
plt.close()


# PERMANOVA (Custom)

def permanova(distance_matrix, groups, permutations=999):
    unique = np.unique(groups)
    group_index = {g: np.where(groups == g)[0] for g in unique}

    ss_between = sum(
        len(group_index[g]) * (distance_matrix[np.ix_(group_index[g], group_index[g])].mean() -
                               distance_matrix.mean())
        for g in unique
    )

    ss_total = ((distance_matrix - distance_matrix.mean())**2).sum()

    f_stat = ss_between / (ss_total - ss_between)

    count = 0
    for _ in range(permutations):
        perm = np.random.permutation(groups)
        gi_perm = {g: np.where(perm == g)[0] for g in unique}
        ss_between_perm = sum(
            len(gi_perm[g]) * (distance_matrix[np.ix_(gi_perm[g], gi_perm[g])].mean() -
                               distance_matrix.mean())
            for g in unique
        )
        f_perm = ss_between_perm / (ss_total - ss_between_perm)
        if f_perm >= f_stat:
            count += 1

    p = (count + 1) / (permutations + 1)
    return f_stat, p

f_stat, p_value = permanova(D, y.values)
with open(f"{OUTDIR}/PERMANOVA_results.txt", "w") as f:
    f.write(f"PERMANOVA pseudo-F = {f_stat}\np = {p_value}\n")


# MACHINE LEARNING

X = X_clr

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    "XGBoost": XGBClassifier(eval_metric="logloss", learning_rate=0.1, n_estimators=300),
    "RandomForest": RandomForestClassifier(n_estimators=300),
    "Logistic": LogisticRegression(max_iter=2000),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(),
    "MLP": MLPClassifier(hidden_layer_sizes=(50,50), max_iter=1000)
}

results = []

plt.figure(figsize=(7,6))

for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    proba = model.predict_proba(X_test_s)[:,1]
    
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    fpr, tpr, _ = roc_curve(y_test, proba)
    
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.2f})")

    results.append([name, acc, auc])
    joblib.dump(model, f"{OUTDIR}/{name}_model.pkl")

plt.plot([0,1],[0,1],'k--')
plt.legend()
plt.title("ROC Comparison")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/ROC_comparison.png")
plt.close()

pd.DataFrame(results, columns=["Model","Accuracy","ROC_AUC"]).to_csv(
    f"{OUTDIR}/model_performance.csv", index=False
)

# SHAP (XGBoost)

xgb = models["XGBoost"]
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test_s)

shap.summary_plot(shap_values, X_test, show=False)
plt.savefig(f"{OUTDIR}/SHAP_summary.png")
plt.close()

print("Pipeline complete! All results saved in:", OUTDIR)
